# ReACT Agent from Scratch in Python Programming

In [ ]:
!pip install langsmith langchain_classic langchain langchain_community langchain_groq langchain_core pydantic chromadb langchain-tavily langchain_huggingface youtube_search wikipedia httpx

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 109.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 106.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 88.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 7.5 MB/s eta 0:00:00
   ━━━━━

In [ ]:
import os
os.environ['LANGSMITH_PROJECT'] = "langchain-agent"

In [ ]:
from google.colab import userdata
os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
os.environ['LANGSMITH_API_KEY'] = userdata.get('LANGCHAIN_API_KEY')
os.environ['LANGSMITH_ENDPOINT']= "https://api.smith.langchain.com"
os.environ['LANGSMITH_TRACING'] = "true"
os.environ["TAVILY_API_KEY"]= userdata.get("TAVILY_API_KEY")
os.environ["SERPER_API_KEY"]= userdata.get("SERPER_API_KEY")

In [ ]:
from langchain_groq import ChatGroq
llm=ChatGroq(model_name="meta-llama/llama-4-scout-17b-16e-instruct")

In [ ]:
llm.invoke("Hi").content

"Hi! It's nice to meet you. Is there something I can help you with, or would you like to chat?"

In [ ]:
message = [{"role":"system", "content":"You are a helpful assistant"},
           {"role":"user", "content":"Hey! How are you?"}]

In [ ]:
result = llm.invoke(message)
print(result.content)


I'm doing well, thanks for asking! I'm a large language model, so I don't have feelings like humans do, but I'm always happy to chat with you and help with any questions or topics you'd like to discuss. How about you? How's your day going?


In [ ]:
class Chatbot:
  def __init__(self, system=''):
    self.system = system
    self.message = []
    if self.system:
      self.message.append({"role":"system", "content":system})
  def __call__(self, message):
    self.message.append({"role":"user", "content":message})
    result = self.execute()
    self.message.append({"role":"assistant", "content":result})
    return result

  def execute(self):
    llm = ChatGroq(model_name="meta-llama/llama-4-scout-17b-16e-instruct")
    result = llm.invoke(self.message)
    return result.content

In [ ]:
bot = Chatbot("You are a helpful assistant")

In [ ]:
bot("Hey! How are you?")

"I'm just a language model, I don't have feelings like humans do, but I'm functioning properly and ready to help you with any questions or tasks you have! How can I assist you today?"

In [ ]:
bot.message

[{'role': 'system', 'content': 'You are a helpful assistant'},
 {'role': 'user', 'content': 'Hey! How are you?'},
 {'role': 'assistant',
  'content': "I'm just a language model, I don't have feelings like humans do, but I'm functioning properly and ready to help you with any questions or tasks you have! How can I assist you today?"}]

In [ ]:
bot.execute()

''

In [ ]:
prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop your output an Answer.
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.


Your available actions are:
calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point
syntax if necessary

wikipedia:
e.g. wikipedia: Django
Returns a summary from searching Wikipedia

simon_blog_search:
e.g. simon_blog_search: Python Programming
Search Simon's blog for that term

Example session:
Question: What is the capital of France?
Thought: I should look up France on Wikipedia
Action: wikipedia: France
PAUSE

You will be called again with this:
Observation: France is a country. The capital is Paris.

You then output:
Answer: The capital of France is Paris

Please Note: if you get basic conversation questions like "hi","hello","how are you?",\n
you have to answer "hi","hello","i am good".
""".strip()

## Understand the Regular expression

Pattern Breakdown:

- ^: This matches the start of a string. It means the string must begin with what follows.
- Action:: This is a literal match. It means the string must have the text "Action:" at the beginning.
- (\w+):
    1. The parentheses () define a capture group. This allows you to extract part of the string that matches this section.
    2. \w+ matches one or more word characters (letters, digits, and underscores). This will capture a word that follows "Action:".
- :: This is a literal colon that separates the word matched by (\w+) from the rest of the string.
- (.*):
    1. This is another capture group, where .* matches any character (.) zero or more times (*), which means it captures everything that comes after the second colon.

### What does it do?
This regex is looking for a string that:

1. Starts with the text "Action:".
2. Has a word right after it, separated by a colon.
3. Then, after another colon, it captures everything that follows.

In [ ]:
import re
action_re = re.compile('^Action: (\w+): (.*)')

<>:2: SyntaxWarning: invalid escape sequence '\w'
<>:2: SyntaxWarning: invalid escape sequence '\w'
/tmp/ipykernel_17684/952804720.py:2: SyntaxWarning: invalid escape sequence '\w'
  action_re = re.compile('^Action: (\w+): (.*)')


In [ ]:
import re

# Compiling the regular expression pattern
action_re = re.compile('^Action: (\w+): (.*)')

# Sample strings
text1 = "Action: Move: North"
text2 = "Action: Jump: High"
text3 = "Error: Not an Action"

<>:4: SyntaxWarning: invalid escape sequence '\w'
<>:4: SyntaxWarning: invalid escape sequence '\w'
/tmp/ipykernel_17684/1117371823.py:4: SyntaxWarning: invalid escape sequence '\w'
  action_re = re.compile('^Action: (\w+): (.*)')


In [ ]:
# Testing the pattern
match1 = action_re.match(text1)

In [ ]:
match1

<re.Match object; span=(0, 19), match='Action: Move: North'>

In [ ]:
match1.group(1)

'Move'

In [ ]:
match1.group(2)

'North'

In [ ]:
match2 = action_re.match(text2)

In [ ]:
match2

<re.Match object; span=(0, 18), match='Action: Jump: High'>

In [ ]:
match3 = action_re.match(text3)

In [ ]:
match3

In [ ]:
# Extracting the match group
if match1:
  print(f'Matched: {match1.group(1)}, {match1.group(2)}')

Matched: Move, North


In [ ]:
if match2:
  print(f'Matched: {match2.group(1)}, {match2.group(2)}')

Matched: Jump, High


In [ ]:
if match3:
  print("No match found")

In [ ]:
import httpx
def wikipedia(query):
    response = httpx.get("https://en.wikipedia.org/w/api.php", params={
        "action": "query",
        "list": "search",
        "srsearch": query,
        "format": "json"
    })
    return response

In [ ]:
wikipedia('Quantum computing')

<Response [403 Forbidden]>

In [ ]:
import httpx
def simon_blog_search(query):
  response = httpx.get("https://datasette.simonwillison.net/simonwillisonblog.json", params={
        "sql": """
        select
          blog_entry.title || ': ' || substr(html_strip_tags(blog_entry.body), 0, 1000) as text,
          blog_entry.created
        from
          blog_entry join blog_entry_fts on blog_entry.rowid = blog_entry_fts.rowid
        where
          blog_entry_fts match escape_fts(:q)
        order by
          blog_entry_fts.rank
        limit
          1
        """.strip(),
        "_shape": "array",
        "q": query,
    })
  return response.json()[0]["text"]

In [ ]:
def calculate(number):
  return eval(number)

In [ ]:
calculate("2+2")

4

In [ ]:
known_actions = {
    "wikipedia":wikipedia,
    "calculate": calculate,
    "simon_blog_search": simon_blog_search,
}


In [ ]:
bot = Chatbot(prompt)

In [ ]:
next_prompt="Tell me about quantum computing from Wikipedia."

In [ ]:
result = bot(next_prompt)
print(result)

Thought: I should look up quantum computing on Wikipedia to get an overview of the topic.

Action: wikipedia: Quantum computing

PAUSE


In [ ]:
actions = [action_re.match(a) for a in result.split('\n') if action_re.match(a)]

In [ ]:
actions

[<re.Match object; span=(0, 36), match='Action: wikipedia: Quantum computing'>]

In [ ]:
action, action_input = actions[0].groups()

In [ ]:
action_input

'Quantum computing'

In [ ]:
action

'wikipedia'

In [ ]:
known_actions[action]

<function __main__.wikipedia(query)>

In [ ]:
known_actions[action]

<function __main__.wikipedia(query)>

In [ ]:
action_input

'Quantum computing'

In [ ]:
known_actions[action]('Quantum computing')

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [60]:
def query(question, max_turns=5):
  i = 0
  bot = Chatbot(prompt)
  next_prompt = question
  while i < max_turns:
    i += 1
    result = bot(next_prompt)
    print(result)
    actions = [action_re.match(a) for a in result.split('\n') if action_re.match(a)]
    if actions:
      action, action_input = actions[0].groups()
      if action not in known_actions:
        raise Exception(f"Unknown action: {action}: {action_input}")
      print(" -- running {} {}".format(action, action_input))
      observation = known_actions[action](action_input)
      print("Observation:", observation)
      next_prompt = f"Observation: {observation}"
    else:
      return result

In [61]:
query('hello')

hello


'hello'

In [62]:
query('how are you?')

Answer: I am good.


'Answer: I am good.'

In [63]:
query('What are you doing?')

Thought: The user is asking what I'm doing, which seems like a basic conversation question.

Action: None needed, I'll respond directly.

Answer: I'm here to help answer questions using a loop of thought, action, pause, and observation.


"Thought: The user is asking what I'm doing, which seems like a basic conversation question.\n\nAction: None needed, I'll respond directly.\n\nAnswer: I'm here to help answer questions using a loop of thought, action, pause, and observation."

In [64]:
print(query("Fifteen * twenty five"))

Thought: I need to calculate the product of 15 and 25.
Action: calculate: 15 * 25
PAUSE
 -- running calculate 15 * 25
Observation: 375
Answer: 375
Answer: 375


In [65]:
query('Has simon written about AI?')

Thought: I should search Simon's blog for AI to see if he has written about it.

Action: simon_blog_search: AI
PAUSE
 -- running simon_blog_search AI
Observation: Talking AI and jobs with Natasha Zouves for News Nation: I was interviewed by News Nation's Natasha Zouves about the very complicated topic of how we should think about AI in terms of threatening our jobs and careers. I previously talked with Natasha two years ago about Microsoft Bing.

I'll be honest: I was nervous about this one. I'm not an economist and I didn't feel confident talking about this topic!

I do find the challenge of making recent advances in AI and LLMs accessible to a general audience absolutely fascinating though, so I took the risk and agreed to the interview.

I think it came out very well. The full hour long video is now available on the News Nation YouTube channel, or as an audio podcast on iTunes or on Spotify.

 

I made my own transcript of the video (using MacWhisper) and fed it into the new Claude 

"Thought: The observation shows that Simon has indeed written about AI, specifically about an interview he had with News Nation about AI and its impact on jobs.\n\nAction: None needed\nObservation: None needed\n\nAnswer: Yes, Simon has written about AI. He was interviewed by News Nation's Natasha Zouves about the topic of AI and its impact on jobs and careers. The interview is available on News Nation's YouTube channel, as an audio podcast on iTunes or Spotify."